# 1. Introduction

The following notebook contains the data preparation for the subsequent analysis. 
The data consists of two datasets: information on electric vehicle charging sessions and the corresponding weather data.
Before the data can be used, it must be checked and, if necessary, adjusted. The goal is to obtain a cleaned, consistent dataset so that we have a suitable basis for further analysis. This is achieved through the following steps:

   - Data Assessment
   - Duplicates
   - Missing Values
   - Data Types
   - Data Cleaning & Filtering
   - Feature Engineering
   - Restructuring Data
   - Final Data Quality Check
   - Train-Test Split


# 2. Import Libraries & Load Data 
## 2.1 Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2.2 Load Data

In [3]:
sessions = pd.read_csv("../data/charging_sessions.csv")
weather = pd.read_csv("../data/weather_burbank_airport.csv")
display(sessions.head())
display(weather.head())

,Unnamed: 0,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,0,5e23b149f9af8b5fe4b973cf,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,194.0,"[{'WhPerMile': 250, 'kWhRequested': 25.0, 'mil..."
1,1,5e23b149f9af8b5fe4b973d0,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1_1_193_825_2020-01-02 13:36:49.599853,1,AG-1F01,1-1-193-825,America/Los_Angeles,4275.0,"[{'WhPerMile': 280, 'kWhRequested': 70.0, 'mil..."
2,2,5e23b149f9af8b5fe4b973d1,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1_1_193_829_2020-01-02 13:56:35.214993,1,AG-1F03,1-1-193-829,America/Los_Angeles,344.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
3,3,5e23b149f9af8b5fe4b973d2,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1_1_193_820_2020-01-02 13:59:58.309319,1,AG-1F04,1-1-193-820,America/Los_Angeles,1117.0,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
4,4,5e23b149f9af8b5fe4b973d3,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1_1_193_819_2020-01-02 14:00:00.779967,1,AG-1F06,1-1-193-819,America/Los_Angeles,334.0,"[{'WhPerMile': 400, 'kWhRequested': 16.0, 'mil..."


,city,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
0,Burbank,2018-01-01 08:53:00,9.0,33.0,Fair,991.75,9.0,0.0,8.0
1,Burbank,2018-01-01 09:53:00,9.0,33.0,Fair,992.08,0.0,0.0,9.0
2,Burbank,2018-01-01 10:53:00,9.0,21.0,Haze,992.08,0.0,0.0,9.0
3,Burbank,2018-01-01 11:53:00,9.0,29.0,Partly Cloudy,992.08,0.0,0.0,9.0
4,Burbank,2018-01-01 12:53:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0


# 3. Data Preparation
## 3.1 Data Assessment

In [4]:
# Overview of charging sessions
print("Charging Sessions:")
print(sessions.info())
print("Shape:", sessions.shape)

# Overview of weather data
print("\nWeather Data:")
print(weather.info())
print("Shape:", weather.shape)

Charging Sessions:
<class 'pandas.DataFrame'>
RangeIndex: 66450 entries, 0 to 66449
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        66450 non-null  int64  
 1   id                66450 non-null  str    
 2   connectionTime    66450 non-null  str    
 3   disconnectTime    66450 non-null  str    
 4   doneChargingTime  62362 non-null  str    
 5   kWhDelivered      66450 non-null  float64
 6   sessionID         66450 non-null  str    
 7   siteID            66450 non-null  int64  
 8   spaceID           66450 non-null  str    
 9   stationID         66450 non-null  str    
 10  timezone          66450 non-null  str    
 11  userID            49187 non-null  float64
 12  userInputs        49187 non-null  str    
dtypes: float64(2), int64(2), str(9)
memory usage: 31.5 MB
None
Shape: (66450, 13)

Weather Data:
<class 'pandas.DataFrame'>
RangeIndex: 29244 entries, 0 to 29243
Data columns (t

## 3.2 Duplicates

First, the session IDs are checked for duplicate rows, and then the system checks to see if there are any duplicate session IDs. This process is then repeated with the weather data.

In [5]:


print("Charging Sessions:")
print("Duplicate rows:", sessions.duplicated().sum())
print("Duplicate Session-IDs:", sessions.duplicated(subset=["sessionID"]).sum())
dups = sessions[sessions.duplicated(subset=["sessionID"], keep=False)].sort_values("sessionID")
print("Affected Rows:", len(dups))
print("Affected IDs:", dups["sessionID"].nunique())

print("\nWeather Data:")
print("Duplicate rows:", weather.duplicated().sum())
print("Duplicate Weather Timestamps:", weather.duplicated(subset=["timestamp"]).sum())

Charging Sessions:
Duplicate rows: 0
Duplicate Session-IDs: 1413
Affected Rows: 2826
Affected IDs: 1413

Weather Data:
Duplicate rows: 0
Duplicate Weather Timestamps: 0


No duplicates were found in the weather data. However, entries with the same ID were found in “Charging sessions.” The next step is to check whether these are export errors. For this purpose, the rows are compared to determine whether they are identical except for the Unnamed: 0 column ( index saved during the CSV export).


In [6]:
columns_to_compare = sessions.columns.drop("Unnamed: 0")

print("Duplicates without Unnamed: 0:",
      sessions.duplicated(subset=columns_to_compare).sum())

Duplicates without Unnamed: 0: 1413


These lines are, in fact, export errors, so they will be deleted.

In [7]:
sessions = sessions.drop(columns=["Unnamed: 0"])
sessions = sessions.drop_duplicates()
print("Duplicate rows:", sessions.duplicated().sum())

Duplicate rows: 0


## 3.3 Missing values
In the next step the datasets are checked for missing values.

In [8]:
# Charging sessions
print("Charging Sessions:")
display(sessions.isnull().sum().sort_values(ascending=False))

# Weather data
print("Weather Data:")
display(weather.isnull().sum().sort_values(ascending=False))


Charging Sessions:


userInputs          17215
userID              17215
doneChargingTime     4087
id                      0
disconnectTime          0
connectionTime          0
kWhDelivered            0
sessionID               0
spaceID                 0
siteID                  0
timezone                0
stationID               0
dtype: int64

Weather Data:


windspeed                  86
felt_temperature           26
temperature                25
cloud_cover_description    20
cloud_cover                20
pressure                    8
city                        0
timestamp                   0
precipitation               0
dtype: int64

It is notable that `userID` and `userInputs` are missing with equal frequency. Therefore, it is checked whether this occurs in the same rows.
If this is the case, it could be due to unregistered users.

In [9]:
print("Is it the same rows:",
      (sessions["userID"].isnull() == sessions["userInputs"].isnull()).all())

Is it the same rows: True


Since these are the same columns, it is assumed that they are unregistered users. A flag  is introduced for this purpose.


In [10]:
sessions["is_registered"] = sessions["userID"].notnull()
print(sessions["is_registered"].value_counts())


is_registered
True     47822
False    17215
Name: count, dtype: int64


In the next step, `doneChargingTime` will be examined.

In [11]:
print(sessions.groupby(sessions["doneChargingTime"].isnull())["kWhDelivered"].describe())

                    count       mean        std    min     25%     50%  \
doneChargingTime                                                         
False             60950.0  11.583268  10.153601  0.501  5.0130   8.991   
True               4087.0  13.986472  11.767162  0.502  5.7845  10.899   

                     75%         max  
doneChargingTime                      
False             13.984  108.797242  
True              18.145   77.700000  


In [13]:
sessions["doneChargingTime"] = sessions["doneChargingTime"].fillna(sessions["disconnectTime"])

still_missing = sessions[sessions["doneChargingTime"].isna()]
print(f"{len(still_missing)} Zeilen ohne doneChargingTime")
still_missing

0 Zeilen ohne doneChargingTime


,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs,is_registered


In [14]:
print(sessions.groupby(sessions["doneChargingTime"] == sessions["disconnectTime"])["kWhDelivered"].describe())

         count       mean        std    min      25%     50%      75%  \
False  60945.0  11.583387  10.153710  0.501  5.01300   8.991  13.9840   
True    4092.0  13.981767  11.764582  0.502  5.78775  10.885  18.1435   

              max  
False  108.797242  
True    77.700000  


In [16]:
dur_h = (pd.to_datetime(sessions["doneChargingTime"]) - pd.to_datetime(sessions["connectionTime"])).dt.total_seconds() / 3600

print(dur_h[dur_h > 0].groupby(sessions["doneChargingTime"] == sessions["disconnectTime"]).describe())

         count      mean       std       min       25%       50%       75%  \
False  60916.0  3.526389  3.064319  0.000833  1.619722  2.760833  4.726250   
True    4092.0  4.742509  4.086069  0.034444  1.029236  3.878889  8.393889   

              max  
False  200.015833  
True    57.457222  


### TODO: Shoudl we instead use an average funktion to replace the values? Using the disconnectTime is about an hour longer than the mean. This would of course reduce variance but it might increase usefullness for analysis such as the clustering

The `doneChargingTime` value is missing for approximately 6.3% of the sessions. A comparison of the energy charged shows that these sessions do not differ significantly from the rest (median 10.9 vs. 9.0 kWh, minimum approximately 0.5 kWh in each case). These are therefore regular charging processes for which the end of charging was not recorded. The values are left blank so as not to skew the charging duration. It will be calculated later only for sessions with available values.

In addition to the columns already examined, the weather data also contain missing values, though only a few (at most 86 per column; see above). Since the weather data constitute a time series, the numerical columns are sorted by timestamp and linearly interpolated. The cloud cover value is filled in with the last known value.

In [17]:
weather["timestamp"] = pd.to_datetime(weather["timestamp"])
weather = weather.sort_values("timestamp").reset_index(drop=True)

num_cols = ["windspeed", "felt_temperature", "temperature", "cloud_cover", "pressure"]
weather[num_cols] = weather[num_cols].interpolate(method="linear")
weather["cloud_cover_description"] = weather["cloud_cover_description"].ffill()

print(weather.isnull().sum())

city                       0
timestamp                  0
temperature                0
cloud_cover                0
cloud_cover_description    0
pressure                   0
windspeed                  0
precipitation              0
felt_temperature           0
dtype: int64


## 3.4 Data Types
In this step, the data types are reviewed and checked for validity. They may be adjusted if necessary.

In [18]:
print("Charging Sessions - Data Types:")
print(sessions.dtypes)
print()
print("Weather Data - Data Types:")
print(weather.dtypes)

Charging Sessions - Data Types:
id                      str
connectionTime          str
disconnectTime          str
doneChargingTime        str
kWhDelivered        float64
sessionID               str
siteID                int64
spaceID                 str
stationID               str
timezone                str
userID              float64
userInputs              str
is_registered          bool
dtype: object

Weather Data - Data Types:
city                                  str
timestamp                  datetime64[us]
temperature                       float64
cloud_cover                       float64
cloud_cover_description               str
pressure                          float64
windspeed                         float64
precipitation                     float64
felt_temperature                  float64
dtype: object


In [19]:
time_cols = ["connectionTime", "disconnectTime", "doneChargingTime"]
for col in time_cols:
    sessions[col] = pd.to_datetime(sessions[col], utc=True)


sessions["userID"] = sessions["userID"].astype("Int64")


sessions["siteID"] = sessions["siteID"].astype("category")

print(sessions.dtypes)

id                                  str
connectionTime      datetime64[us, UTC]
disconnectTime      datetime64[us, UTC]
doneChargingTime    datetime64[us, UTC]
kWhDelivered                    float64
sessionID                           str
siteID                         category
spaceID                             str
stationID                           str
timezone                            str
userID                            Int64
userInputs                          str
is_registered                      bool
dtype: object


The time columns of the charging sessions were stored as text and have been converted to datetime (UTC). userID was stored as a float because of the missing values and has been converted to a nullable integer. siteID is an identifier, not a numeric measure, and was converted to a categorical column. The weather timestamp was already converted during the interpolation step.

## 3.5 Data Cleaning & Filtering

Drop Unnecessary columns and rows

### Remove Negative Charging Times

In [22]:
dur_h = (pd.to_datetime(sessions["doneChargingTime"]) - pd.to_datetime(sessions["connectionTime"])).dt.total_seconds() / 3600

neg = sessions[dur_h < 0].assign(dauer_min=dur_h[dur_h < 0] * 60)
print(f"{len(neg)} Sessions mit negativer Dauer")
neg[["spaceID", "connectionTime", "doneChargingTime", "disconnectTime", "kWhDelivered", "dauer_min"]].sort_values("dauer_min").head(20)

27 Sessions mit negativer Dauer


,spaceID,connectionTime,doneChargingTime,disconnectTime,kWhDelivered,dauer_min
44041,CA-307,2018-11-04 03:56:17+00:00,2018-11-04 03:14:55+00:00,2018-11-04 04:15:00+00:00,0.521000,-41.366667
44042,CA-303,2018-11-04 04:33:19+00:00,2018-11-04 03:52:29+00:00,2018-11-04 04:52:37+00:00,2.051000,-40.833333
44040,CA-495,2018-11-04 02:10:08+00:00,2018-11-04 01:40:31+00:00,2018-11-04 02:55:22+00:00,0.988647,-29.616667
27740,CA-322,2019-09-05 18:44:57+00:00,2019-09-05 18:43:57+00:00,2019-09-06 00:55:19+00:00,1.067230,-1.000000
41263,CA-494,2018-09-27 22:58:45+00:00,2018-09-27 22:57:45+00:00,2018-09-28 21:58:26+00:00,4.443405,-1.000000
42102,CA-493,2018-10-08 19:14:32+00:00,2018-10-08 19:13:32+00:00,2018-10-09 00:54:50+00:00,1.104249,-1.000000
42342,CA-322,2018-10-11 15:58:47+00:00,2018-10-11 15:57:47+00:00,2018-10-11 20:06:02+00:00,0.714337,-1.000000
31287,CA-494,2018-05-04 19:23:52+00:00,2018-05-04 19:22:52+00:00,2018-05-05 00:04:15+00:00,0.912297,-1.000000
42618,CA-322,2018-10-15 15:42:14+00:00,2018-10-15 15:41:14+00:00,2018-10-15 20:39:47+00:00,0.858423,-1.000000
43484,CA-321,2018-10-26 20:27:53+00:00,2018-10-26 20:26:53+00:00,2018-10-27 00:56:04+00:00,0.871274,-1.000000


## TODO: I would like your opinions on if or how we should change these rows
I think we should probably drop these 27 rows.
Also, that we should drop the one row with the overlapping parking spot in which there are no user data. 

## 3.6 Feature Engineering
Add columns, that might be useful (e.g duration, holidays)

In [21]:
# Duration & Holidays

## 3.7 Restructuring Data – Flattening Arrays
Convert array-based columns into individual rows (User Input array)

## 3.8 Final Data Quality Check

## 3.9 Train-Test Split (Cross Validation)
Divide the data into training, validation, and test sets